# Replicação do MTST no ETTh1 (Colab, GPU)
Antes: *Runtime > Alterar tipo de ambiente de execução > GPU* e criar o Secret `GITHUB_TOKEN` (ícone de chave, com acesso ao notebook ligado).

In [1]:
# Clona o repo privado com o token dos Secrets (o token não é gravado no notebook nem em .git/config)
import base64, os, subprocess
from google.colab import userdata
token = userdata.get('GITHUB_TOKEN')
auth = base64.b64encode(f'x-access-token:{token}'.encode()).decode()
if not os.path.exists('/content/mtst-demanda-varejo'):
    subprocess.run(['git', '-c', f'http.extraheader=AUTHORIZATION: basic {auth}', 'clone',
                    'https://github.com/VanthuirMaia/mtst-demanda-varejo.git',
                    '/content/mtst-demanda-varejo'], check=True)
%cd /content/mtst-demanda-varejo
!git log --oneline -3

/content/mtst-demanda-varejo
db08018 (HEAD -> main, origin/main, origin/HEAD) README: pegadinhas do oficial e passo a passo do Colab
e2d34ed Notebook Colab para a replicação ETTh1
c62065c replicacao.py: device automático, 8 threads, etth1_resultados.csv com comparação ao artigo


In [2]:
# torch, pandas e numpy já vêm no Colab; falta só isto
!pip install -q einops yacs

In [3]:
# ETTh1 do repositório público do ETDataset
!mkdir -p data/raw
!wget -q -O data/raw/ETTh1.csv https://raw.githubusercontent.com/zhouhaoyi/ETDataset/main/ETT-small/ETTh1.csv
!wc -l data/raw/ETTh1.csv

17421 data/raw/ETTh1.csv


In [4]:
import torch
assert torch.cuda.is_available(), 'Sem GPU: ative em Ambiente de execução > Alterar tipo'
print(torch.cuda.get_device_name(0))

Tesla T4


In [5]:
# 4 horizontes em sequência (cada um atualiza results/tables/etth1_resultados.csv)
for H in (96, 336, 192, 720):
    !python -u -m src.replicacao --horizonte {H}

[etth1_h96] device: Tesla T4
[etth1_h96] batches treino=32 val=10 teste=10
[etth1_h96] parâmetros: 7,643,760
[etth1_h96] shapes: x=(256, 336, 7) y=(256, 96, 7) saída=(256, 96, 7)
[etth1_h96] época 001 | treino 0.7454 | val 0.9035 | lr 4.00e-06 | 19s *
[etth1_h96] época 002 | treino 0.5921 | val 0.8045 | lr 1.00e-04 | 19s *
[etth1_h96] época 003 | treino 0.5097 | val 0.7323 | lr 1.00e-04 | 20s *
[etth1_h96] época 004 | treino 0.4738 | val 0.7154 | lr 1.00e-04 | 21s *
[etth1_h96] época 005 | treino 0.4534 | val 0.7271 | lr 9.00e-05 | 21s
[etth1_h96] época 006 | treino 0.4409 | val 0.7170 | lr 8.10e-05 | 21s
[etth1_h96] época 007 | treino 0.4305 | val 0.7017 | lr 7.29e-05 | 21s *
[etth1_h96] época 008 | treino 0.4223 | val 0.7121 | lr 6.56e-05 | 21s
[etth1_h96] época 009 | treino 0.4144 | val 0.6925 | lr 5.90e-05 | 21s *
[etth1_h96] época 010 | treino 0.4090 | val 0.6994 | lr 5.31e-05 | 21s
[etth1_h96] época 011 | treino 0.4039 | val 0.7033 | lr 4.78e-05 | 21s
[etth1_h96] época 012 | trei

In [6]:
import pandas as pd
pd.read_csv('results/tables/etth1_resultados.csv')

,horizonte,mse,mae,melhor_epoca,epocas_rodadas,melhor_val,tempo_min,device,mse_artigo,mse_dif_abs,mse_dif_rel_pct,mae_artigo,mae_dif_abs,mae_dif_rel_pct
0,96,0.360881,0.390447,12,32,0.681629,11.264140,Tesla T4,0.358,0.002881,0.804649,0.390,0.000447,0.114697
1,192,0.402774,0.415717,8,28,0.938226,9.633420,Tesla T4,0.396,0.006774,1.710660,0.414,0.001717,0.414779
2,336,0.395669,0.423204,5,25,1.204218,8.625584,Tesla T4,0.391,0.004669,1.194205,0.420,0.003204,0.762787
3,720,0.436071,0.461192,4,24,1.487882,8.004806,Tesla T4,0.430,0.006071,1.411752,0.457,0.004192,0.917244


In [7]:
# Compacta resultados e logs para download
!zip -q -r etth1_resultados.zip results/tables/etth1_*
from google.colab import files
files.download('etth1_resultados.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>